# 04 · DFT, FFT e amostragem

**Objetivo pedagógico:** Comparar transformadas com convenção consistente, reconstruir sinais e identificar aliasing.

**Luigi Scofano de Araujo · USP 13680334**

Reelaboração de estudo de um tema presente no acervo recuperado de 2025. Esta edição foi preparada em 2026 com assistência de IA. Não é uma cópia autenticada de uma entrega nem uma afirmação de conclusão das questões originais. Os arquivos recuperados estão em `acervo/`, com créditos e limitações em `docs/PROCEDENCIA.md`.

Execute as células em ordem, em kernel novo. Os dados são locais/sintéticos e as conclusões numéricas são verificadas por asserções; gráficos sozinhos não certificam convergência.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd()/"numericos").is_dir() else Path.cwd().parent
if not (ROOT/"numericos").is_dir():
    raise RuntimeError("Execute a partir da raiz do repositório ou de notebooks/.")
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=7, suppress=True)
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True})
FIGURAS = ROOT/"assets"/"figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)
def mostrar(nome):
    plt.tight_layout()
    plt.savefig(FIGURAS/(nome+".svg"), bbox_inches="tight", metadata={"Date": None})
    plt.show()
    plt.close("all")

## 1. Convenção
A transformada direta usa sinal negativo e não divide por N; a inversa divide por N. Essa é a convenção padrão `backward` de NumPy. No acervo há DFT dividida por N e FFT sem essa divisão; compará-las diretamente criava uma diferença artificial de escala.

In [2]:
from numericos.fourier import dft,fft,ifft
N=128;fs=128.;t=np.arange(N)/fs
signal=np.sin(2*np.pi*5*t)+0.4*np.cos(2*np.pi*17*t)
D=dft(signal);F=fft(signal)
assert np.allclose(D,F,atol=1e-10)
assert np.allclose(F,np.fft.fft(signal),atol=1e-10)
assert np.allclose(ifft(F),signal,atol=1e-12)
frequencies=np.fft.fftfreq(N,1/fs)
plt.stem(frequencies[:N//2],2*abs(F[:N//2])/N)
plt.xlabel("frequência (Hz)");plt.ylabel("amplitude unilateral")
mostrar("04_espectro")
print("máxima diferença DFT/FFT:",np.max(abs(D-F)))

máxima diferença DFT/FFT: 1.3095903634194824e-12


## 2. Aliasing
Amostras de 27 Hz a 32 Hz não distinguem esse sinal de uma frequência de 5 Hz com fase/sinal apropriado. Um espectro amostrado não revela por si só a frequência analógica original. O gráfico unilateral anterior tem interpretação simples para esses cossenos/senos nas frequências da malha; DC e Nyquist requerem cuidado no fator 2.

In [3]:
fs=32.;t=np.arange(64)/fs
high=np.sin(2*np.pi*27*t)
alias=-np.sin(2*np.pi*5*t)
assert np.allclose(high,alias,atol=1e-12)
plt.plot(t,high,"o",label="amostras 27 Hz")
plt.plot(t,alias,"-",label="alias −sen(2π·5t)")
plt.xlabel("tempo (s)");plt.ylabel("amplitude");plt.legend()
mostrar("04_aliasing")

## 3. Custo e medição
A DFT densa armazena N² complexos: para N=32768, apenas a matriz complex128 exige 16 GiB, além de temporários. A edição de estudo limita a comparação a N≤512 e o módulo rejeita DFT densa acima de 2048. Tempos dependem da máquina e não são testes de performance com limiares fixos.

In [4]:
from time import perf_counter
rows=[]
for N in [32,64,128,256,512]:
    values=np.cos(np.arange(N)/N*2*np.pi*3)
    measurements=[]
    for transform in [dft,fft,np.fft.fft]:
        start=perf_counter();result=transform(values);measurements.append(perf_counter()-start)
        assert np.allclose(result,np.fft.fft(values),atol=1e-9)
    rows.append([N,*measurements])
print("N      DFT(s)       FFT didática(s)    NumPy(s)")
for row in rows:print(f"{row[0]:4} {row[1]:12.3e} {row[2]:16.3e} {row[3]:12.3e}")
try:fft([1,2,3])
except ValueError as error:print("Falha esperada:",error)

N      DFT(s)       FFT didática(s)    NumPy(s)
  32    1.342e-04        2.047e-04    6.550e-06
  64    2.544e-04        3.005e-04    5.618e-06
 128    4.512e-04        6.626e-04    7.492e-06
 256    2.657e-03        2.247e-03    9.985e-06
 512    8.366e-03        3.417e-03    1.166e-05
Falha esperada: A FFT didática radix-2 exige tamanho potência de dois.


## Para estudar
Compare erros com e sem normalização consistente. Use um comprimento não potência de dois e explique a limitação da implementação radix-2. Não confunda o custo assintótico menor com ser mais rápido em todo N: recursão Python tem custo significativo.

## Figuras desta execução

![04_aliasing](../assets/figuras/04_aliasing.svg)

![04_espectro](../assets/figuras/04_espectro.svg)